# Recommendation / 03 — Chronological split

Preserve the CSV’s fixed chronological split. Never randomly distribute copies of a monthly observation or customer request across sets.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
df = pd.read_csv(w.DATA / "recommendation.csv")
train = df[df.dataset_split == "train"].copy()
validation = df[df.dataset_split == "validation"].copy()
test = df[df.dataset_split == "test"].copy()
display(df.groupby("dataset_split").agg(rows=("year","size"), first_year=("year","min"), last_year=("year","max")))
assert train.year.max() < validation.year.min() < test.year.min()
key = "destination_month_id" if "recommendation" == "crowd" else "request_id"
assert not set(train[key]) & set(validation[key])
assert not set(train[key]) & set(test[key])
assert not set(validation[key]) & set(test[key])
print("Chronological and grouping checks passed.")

Chronological and grouping checks passed.


,rows,first_year,last_year
dataset_split,,,
test,2880,2024,2025
train,10080,2016,2022
validation,1440,2023,2023


Train: 2016–2022. Validation: 2023. Test: 2024–2025. The test set is only used after selecting the winner. Busyness categories were generated using fixed simulator reference levels, not quantiles learned from test records.